# XTTS-v2 (Coqui) — Toy Voice-Cloning TTS

In [ ]:
import mathimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoader

In [ ]:
class Config:    text_vocab_size = 256    mel_vocab_size = 1024    n_mels = 80    d_model = 128    n_heads = 4    n_text_layers = 2    n_decoder_layers = 3    speaker_emb_dim = 128    block_size = 200    dropout = 0.1cfg = Config()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class PositionalEncoding(nn.Module):    def __init__(self, d_model, max_len):        super().__init__()        pe = torch.zeros(max_len, d_model)        position = torch.arange(0, max_len).unsqueeze(1).float()        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))        pe[:, 0::2] = torch.sin(position * div_term)        pe[:, 1::2] = torch.cos(position * div_term)        self.register_buffer("pe", pe.unsqueeze(0))    def forward(self, x):        return x + self.pe[:, :x.size(1)]

In [ ]:
class SpeakerEncoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.conv1 = nn.Conv1d(cfg.n_mels, cfg.d_model, kernel_size=3, stride=2, padding=1)        self.conv2 = nn.Conv1d(cfg.d_model, cfg.d_model, kernel_size=3, stride=2, padding=1)        self.gru = nn.GRU(cfg.d_model, cfg.speaker_emb_dim, batch_first=True)        self.proj = nn.Linear(cfg.speaker_emb_dim, cfg.speaker_emb_dim)    def forward(self, ref_mel):        x = ref_mel.transpose(1, 2)        x = F.relu(self.conv1(x))        x = F.relu(self.conv2(x))        x = x.transpose(1, 2)        _, h = self.gru(x)        emb = self.proj(h.squeeze(0))        return F.normalize(emb, dim=-1)

In [ ]:
class TextEncoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.embedding = nn.Embedding(cfg.text_vocab_size, cfg.d_model, padding_idx=0)        self.pos_enc = PositionalEncoding(cfg.d_model, cfg.block_size)        layer = nn.TransformerEncoderLayer(            d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=4 * cfg.d_model,            dropout=cfg.dropout, batch_first=True        )        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_text_layers)    def forward(self, text_tokens):        x = self.embedding(text_tokens)        x = self.pos_enc(x)        return self.encoder(x)

In [ ]:
class GPTMelDecoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.mel_emb = nn.Embedding(cfg.mel_vocab_size, cfg.d_model)        self.speaker_proj = nn.Linear(cfg.speaker_emb_dim, cfg.d_model)        self.pos_enc = PositionalEncoding(cfg.d_model, cfg.block_size)        layer = nn.TransformerDecoderLayer(            d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=4 * cfg.d_model,            dropout=cfg.dropout, batch_first=True        )        self.decoder = nn.TransformerDecoder(layer, num_layers=cfg.n_decoder_layers)        self.head = nn.Linear(cfg.d_model, cfg.mel_vocab_size)    def forward(self, mel_tokens, text_memory, speaker_emb):        t = mel_tokens.size(1)        x = self.mel_emb(mel_tokens) + self.speaker_proj(speaker_emb).unsqueeze(1)        x = self.pos_enc(x)        causal_mask = torch.triu(torch.ones(t, t, device=mel_tokens.device), diagonal=1).bool()        x = self.decoder(x, text_memory, tgt_mask=causal_mask)        return self.head(x)

In [ ]:
class Vocoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.mel_emb = nn.Embedding(cfg.mel_vocab_size, cfg.d_model)        self.upsample1 = nn.ConvTranspose1d(cfg.d_model, cfg.d_model // 2, kernel_size=4, stride=2, padding=1)        self.upsample2 = nn.ConvTranspose1d(cfg.d_model // 2, cfg.d_model // 4, kernel_size=4, stride=2, padding=1)        self.out_conv = nn.Conv1d(cfg.d_model // 4, 1, kernel_size=7, padding=3)    def forward(self, mel_tokens):        x = self.mel_emb(mel_tokens).transpose(1, 2)        x = F.leaky_relu(self.upsample1(x))        x = F.leaky_relu(self.upsample2(x))        waveform = torch.tanh(self.out_conv(x))        return waveform.squeeze(1)

In [ ]:
class XTTS(nn.Module):    def __init__(self, cfg):        super().__init__()        self.speaker_encoder = SpeakerEncoder(cfg)        self.text_encoder = TextEncoder(cfg)        self.mel_decoder = GPTMelDecoder(cfg)        self.vocoder = Vocoder(cfg)    def forward(self, text_tokens, ref_mel, mel_tokens_input):        speaker_emb = self.speaker_encoder(ref_mel)        text_memory = self.text_encoder(text_tokens)        mel_logits = self.mel_decoder(mel_tokens_input, text_memory, speaker_emb)        return mel_logits, speaker_emb    def synthesize(self, text_tokens, ref_mel, max_len):        speaker_emb = self.speaker_encoder(ref_mel)        text_memory = self.text_encoder(text_tokens)        generated = torch.zeros(text_tokens.size(0), 1, dtype=torch.long, device=text_tokens.device)        for _ in range(max_len - 1):            logits = self.mel_decoder(generated, text_memory, speaker_emb)            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)            generated = torch.cat([generated, next_token], dim=1)        waveform = self.vocoder(generated)        return generated, waveform

In [ ]:
class ToyXTTSDataset(Dataset):    def __init__(self, n_samples, cfg):        self.n_samples = n_samples        self.cfg = cfg    def __len__(self):        return self.n_samples    def __getitem__(self, idx):        text_len = torch.randint(10, 40, (1,)).item()        text_tokens = torch.randint(1, self.cfg.text_vocab_size, (text_len,))        ref_mel = torch.randn(100, self.cfg.n_mels)        mel_len = torch.randint(20, 60, (1,)).item()        mel_tokens = torch.randint(1, self.cfg.mel_vocab_size, (mel_len,))        return text_tokens, ref_mel, mel_tokensdef collate_fn(batch):    texts, ref_mels, mels = zip(*batch)    text_lens = [t.size(0) for t in texts]    mel_lens = [m.size(0) for m in mels]    max_text_len = max(text_lens)    max_mel_len = max(mel_lens)    text_padded = torch.zeros(len(batch), max_text_len, dtype=torch.long)    mel_padded = torch.zeros(len(batch), max_mel_len, dtype=torch.long)    ref_mel_padded = torch.stack(ref_mels)    for i in range(len(batch)):        text_padded[i, :text_lens[i]] = texts[i]        mel_padded[i, :mel_lens[i]] = mels[i]    return text_padded, ref_mel_padded, mel_padded

In [ ]:
dataset = ToyXTTSDataset(n_samples=32, cfg=cfg)loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)model = XTTS(cfg).to(device)optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
def train_epoch(model, loader, optimizer):    model.train()    total_loss = 0.0    for text_tokens, ref_mel, mel_tokens in loader:        text_tokens = text_tokens.to(device)        ref_mel = ref_mel.to(device)        mel_tokens = mel_tokens.to(device)        decoder_input = F.pad(mel_tokens[:, :-1], (1, 0), value=0)        target = mel_tokens        mel_logits, _ = model(text_tokens, ref_mel, decoder_input)        loss = F.cross_entropy(mel_logits.reshape(-1, cfg.mel_vocab_size), target.reshape(-1), ignore_index=0)        optimizer.zero_grad()        loss.backward()        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)

In [ ]:
for epoch in range(5):    avg_loss = train_epoch(model, loader, optimizer)    print(f"epoch {epoch + 1} loss {avg_loss:.4f}")

In [ ]:
model.eval()with torch.no_grad():    text_prompt = torch.randint(1, cfg.text_vocab_size, (1, 20)).to(device)    ref_mel_sample = torch.randn(1, 100, cfg.n_mels).to(device)    generated_tokens, waveform = model.synthesize(text_prompt, ref_mel_sample, max_len=30)    print(generated_tokens.shape, waveform.shape)